In [7]:
import numpy as np
import pandas as pd
import math
import tempfile
import os
from scipy.spatial.distance import cdist
import pulp
import re
import glob
import random

# **Data input**

In [8]:
# ==========================================
# 1. CONFIGURATION
# ==========================================
# Path to the CSV generated by your previous batch run
INPUT_CSV_PATH = "TSP_single_dual_bound_50_cus_results.csv"

# Path to the folder containing the instance .txt files (n50)
# Update this if your files are in a different location
DATASET_FOLDER = r"C:\Users\ACER\Desktop\Code\0.Thesis implementation\2_DIDP_custom_search_guidance_local\Thesis_modified_DIDP\1_TSP_dual_bounds_and_models\Datasets\n50"

# ==========================================
# 2. HELPER FUNCTIONS (Reused)
# ==========================================

def read_tsp_cappart_format(file_path):
    """ Parses the TSP text files to extract N and Cost Matrix. """
    with open(file_path, 'r') as f:
        values = f.read().split()

    iterator = iter(values)
    try:
        n = int(next(iterator))
        c = []
        for i in range(n):
            row = []
            for j in range(n):
                val = float(next(iterator))
                row.append(int(val))
            c.append(row)
        return n, c
    except StopIteration:
        raise ValueError(f"File {file_path} ended unexpectedly.")

# **DIDP model**

In [ ]:
def create_tsp_mtz_relaxed_model(num_locations, travel_cost):
    """
    Creates a TSP Linear Programming Relaxation model with MTZ subtour elimination.
    
    Args:
        n_nodes (int): Number of nodes (n)
        dist_matrix (list of lists): n x n cost matrix
        
    Returns:
        pulp.LpProblem: The defined model
    """
    mdl = pulp.LpProblem("TSP_MTZ_Relaxed", pulp.LpMinimize)

    # Variables
    # x[i, j]: Flow variables (Continuous 0-1)
    # Represents fraction of travel from i to j
    x = {}
    for i in range(num_locations):
        for j in range(num_locations):
            if i != j:
                x[(i, j)] = pulp.LpVariable(f"x_{i}_{j}", 0, 1, pulp.LpContinuous)

    # u[i]: MTZ potential variables (Continuous)
    # Represents the order/sequence of node i in the tour
    # Defined only for nodes 1..n-1 (Node 0 is the start/anchor)
    # Bounds: 1 <= u_i <= n-1
    u = {}
    for i in range(1, num_locations):
        u[i] = pulp.LpVariable(f"u_{i}", 1, num_locations - 1, pulp.LpContinuous)

    # Objective
    # Minimize sum(c_ij * x_ij)
    mdl += pulp.lpSum(travel_cost[i][j] * x[(i, j)] 
                      for i in range(num_locations)
                      for j in range(num_locations) if i != j), "Total_Cost"

    # Constraints
    # Degree Constraints
    # (A) Assignment Constraints (Degree Constraints)
    for k in range(num_locations):
        # Outgoing flow = 1
        mdl += pulp.lpSum(x[(k, j)] for j in range(num_locations) if k != j) == 1, f"Out_{k}"
        # Incoming flow = 1
        mdl += pulp.lpSum(x[(i, k)] for i in range(num_locations) if i != k) == 1, f"In_{k}"

    # MTZ Subtour Elimination
    # (B) MTZ Subtour Elimination Constraints
    # u_i - u_j + n * x_ij <= n - 1
    # Valid for all i, j in {1, ..., n-1}, i != j
    # This prevents cycles that do not include node 0
    for i in range(1, num_locations):
        for j in range(1, num_locations):
            if i != j:
                mdl += u[i] - u[j] + num_locations * x[(i, j)] <= num_locations - 1, f"MTZ_{i}_{j}"

    return mdl

# **Execution**

In [11]:
# ==========================================
# 3. MAIN EXECUTION
# ==========================================

def run_lp_post_processing():
    print(f"📂 Loading results from: {INPUT_CSV_PATH}")
    
    if not os.path.exists(INPUT_CSV_PATH):
        print("❌ Error: CSV file not found.")
        return

    # Load DataFrame
    df = pd.read_csv(INPUT_CSV_PATH)
    
    # Add column if it doesn't exist
    if 'LP Relaxed LB' not in df.columns:
        df['LP Relaxed LB'] = None

    print(f"🚀 Processing LP Relaxation for {len(df)} instances...")

    for index, row in df.iterrows():
        instance_name = row['Instance']
        
        # Check if already computed (skip if so, allows resuming)
        if pd.notna(row.get('LP Relaxed LB')) and row.get('LP Relaxed LB') != "":
             continue

        file_path = os.path.join(DATASET_FOLDER, instance_name)
        
        if not os.path.exists(file_path):
            print(f"⚠️ Warning: Data file not found for {instance_name}. Skipping.")
            continue
            
        print(f"   [{index+1}/{len(df)}] Solving LP for {instance_name}...", end=" ", flush=True)
        
        try:
            # 1. Read Data
            n, c = read_tsp_cappart_format(file_path)
            
            # 2. Build Model
            model = create_tsp_mtz_relaxed_model(n, c)
            
            # 3. Solve (Suppress output)
            # Use standard CBC solver. Time limit 60s to prevent hanging on huge instances
            solver = pulp.PULP_CBC_CMD(msg=False, timeLimit=60) 
            model.solve(solver)
            
            # 4. Extract Result
            if model.status == 1: # 1 = Optimal
                lb = pulp.value(model.objective)
                df.at[index, 'LP Relaxed LB'] = lb
                print(f"Done. LB = {lb:.2f}")
            else:
                stat = pulp.LpStatus[model.status]
                df.at[index, 'LP Relaxed LB'] = f"Unsolved ({stat})"
                print(f"Failed. Status: {stat}")

        except Exception as e:
            print(f"Error: {e}")
            df.at[index, 'LP Relaxed LB'] = "Error"

        # Save periodically (optional safety)
        if index % 5 == 0:
            df.to_csv(INPUT_CSV_PATH, index=False)

    # Final Save
    df.to_csv(INPUT_CSV_PATH, index=False)
    print("\n" + "="*50)
    print(f"✅ Completed. Results updated in {INPUT_CSV_PATH}")
    print("="*50)
    print(df[['Instance', 'Cost', 'LP Relaxed LB']].head())

In [12]:
# Run the function
if __name__ == "__main__":
    run_lp_post_processing()

📂 Loading results from: TSP_single_dual_bound_50_cus_results.csv
🚀 Processing LP Relaxation for 20 instances...
   [1/20] Solving LP for 61.txt... 

Done. LB = 509.12
   [2/20] Solving LP for 53.txt... Done. LB = 423.04
   [3/20] Solving LP for 50.txt... Done. LB = 475.84
   [4/20] Solving LP for 89.txt... Done. LB = 426.04
   [5/20] Solving LP for 31.txt... Done. LB = 401.76
   [6/20] Solving LP for 75.txt... Done. LB = 428.58
   [7/20] Solving LP for 52.txt... Done. LB = 485.72
   [8/20] Solving LP for 33.txt... Done. LB = 510.02
   [9/20] Solving LP for 39.txt... Done. LB = 419.26
   [10/20] Solving LP for 11.txt... Done. LB = 510.42
   [11/20] Solving LP for 14.txt... Done. LB = 504.30
   [12/20] Solving LP for 54.txt... Done. LB = 407.22
   [13/20] Solving LP for 78.txt... Done. LB = 414.22
   [14/20] Solving LP for 59.txt... Done. LB = 434.66
   [15/20] Solving LP for 46.txt... Done. LB = 459.36
   [16/20] Solving LP for 4.txt... Done. LB = 486.32
   [17/20] Solving LP for 79.txt... Done. LB = 477.28
   [18/20] Solving LP for 48.txt... Done. LB = 441.72
   [19/20] Solving LP for 22.txt... Done. LB = 527.38
   [20/20] Solving 